1. Environment and Infrastructure Setup

In [ ]:
from google.colab import userdata

# 1. Configuration
USER_NAME = "mervynzwkoh"
REPO_NAME = "gout-transcriptome-causal"
TOKEN = userdata.get("GithubToken")

# 2. Setup the Git URL
# This tells Colab how to talk to your repo securely
repo_url = f"https://{TOKEN}@github.com/{USER_NAME}/{REPO_NAME}.git"

# 3. Clone the repository
!git clone {repo_url}

# 4. Move into the repo folder
%cd {REPO_NAME}

# 5. Configure Git identity (for your commit history)
!git config --global user.email "mervynzwkoh@gmail.com"
!git config --global user.name "Mervyn Koh"

fatal: destination path 'gout-transcriptome-causal' already exists and is not an empty directory.
/content/gout-transcriptome-causal


In [ ]:
import os

# Create the folders we planned
folders = ['data', 'notebooks', 'src', 'docs']
for folder in folders:
    os.makedirs(folder, exist_ok=True)
    # Create a dummy file in each so Git notices the folder
    with open(f"{folder}/.gitkeep", "w") as f:
        pass

# Integrate remote changes first, then add, commit and push
!git pull origin main --rebase
!git add .
!git commit -m "Initial commit: Established project folder structure and added data filtering logic"
!git push origin main

On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean
To https://github.com/mervynzwkoh/gout-transcriptome-causal.git
 ! [rejected]        main -> main (fetch first)
error: failed to push some refs to 'https://github.com/mervynzwkoh/gout-transcriptome-causal.git'
hint: Updates were rejected because the remote contains work that you do
hint: not have locally. This is usually caused by another repository pushing
hint: to the same ref. You may want to first integrate the remote changes
hint: (e.g., 'git pull ...') before pushing again.
hint: See the 'Note about fast-forwards' in 'git push --help' for details.


2. Data Ingestion and Filtering

In [ ]:
from datasets import load_dataset

# We use streaming=True so we don't run out of memory
dataset_stream = load_dataset("Xaira-Therapeutics/X-Atlas-Orion", streaming=True, split="HEK293T")

# Let's peek at the first "row" (cell) to see what information it contains
first_cell = next(iter(dataset_stream))
print(first_cell.keys())

Resolving data files:   0%|          | 0/109 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/223 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/109 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/223 [00:00<?, ?it/s]

dict_keys(['gene_token_id', 'gene_expression', 'cell_barcode', 'sample', 'num_features', 'guide_target', 'gene_target', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'pass_guide_filter'])


In [ ]:
target_genes = ["ABCG2", "SLC22A12"]
filtered_cells = []

# We'll grab the first 100 cells that match our criteria for this test
print("Searching for gout-related perturbations...")

for cell in dataset_stream:
    # 'perturbation_name' or similar key in Orion identifies the targeted gene
    if cell['gene_target'] in target_genes:
        filtered_cells.append(cell)
        print(f"Found {len(filtered_cells)} cells so far...")

    if len(filtered_cells) >= 35:
        break

print(f"Success! Found {len(filtered_cells)} cells for initial analysis.")

Searching for gout-related perturbations...
Found 1 cells so far...
Found 2 cells so far...
Found 3 cells so far...
Found 4 cells so far...
Found 5 cells so far...
Found 6 cells so far...
Found 7 cells so far...
Found 8 cells so far...
Found 9 cells so far...
Found 10 cells so far...
Found 11 cells so far...
Found 12 cells so far...
Found 13 cells so far...
Found 14 cells so far...
Found 15 cells so far...
Found 16 cells so far...
Found 17 cells so far...
Found 18 cells so far...
Found 19 cells so far...
Found 20 cells so far...
Found 21 cells so far...
Found 22 cells so far...
Found 23 cells so far...
Found 24 cells so far...
Found 25 cells so far...
Found 26 cells so far...
Found 27 cells so far...
Found 28 cells so far...
Found 29 cells so far...
Found 30 cells so far...
Found 31 cells so far...
Found 32 cells so far...
Found 33 cells so far...
Found 34 cells so far...
Found 35 cells so far...
Success! Found 35 cells for initial analysis.


3. Pre-processing Raw Data for Geneformer

In [ ]:
# Software Engineering Tip: We are 'Pre-processing' the data
# to make it ready for the Machine Learning model.

import pandas as pd

def format_for_geneformer(cell_data):
    """
    Combines token IDs and expression values into a ranked list.
    """
    tokens = cell_data['gene_token_id']
    counts = cell_data['gene_expression']

    # We create a dataframe for this specific cell's expression
    cell_df = pd.DataFrame({'token': tokens, 'count': counts})

    # Geneformer often uses rank-based normalization
    cell_df = cell_df.sort_values(by='count', ascending=False)

    return cell_df['token'].tolist()

# Example: Format the first cell we found
formatted_input = format_for_geneformer(filtered_cells[0])
print(f"First cell formatted. Number of expressed genes: {len(formatted_input)}")

First cell formatted. Number of expressed genes: 7785


In [ ]:
# We will use 'num_features' as our initial 'dose' proxy
# to see if more guides correlate with a stronger transcriptomic shift.

doses = [cell['num_features'] for cell in filtered_cells]
print(f"Dose distribution (Number of guides): {set(doses)}")

Dose distribution (Number of guides): {2}
